# Generic Programming and Templates

Templates are code blueprints. You write one generic definition, and during compilation, the compiler stamps out the concrete code for the real types you use. This gives you opportunities for code reuse, type safety, and speed because the generated code is type-specific and optimized before the program runs. This gives you opportunities for code reuse, type-specific and optimized before the program runs. We will apply this idea to functions and classes, and see how the compiler chooses types at compile time, while functions receive values at runtime.

````C++
1| template <typename T>
2| T max_of_three(const T& a, const T& b, const T& c){  // max_of_three is known as specialization
3|      return std::max({a,b,c});    
4| }
````

Considering the previous code, in line 1 `template <typename T>` tells the compiler that when it sees this template used with concrete types, it should use the template as blueprint to generate a new function for those types. That generated function is calle a specialization, meaning the concrete function the compiler emits for a specific type. At each place in your code where you call the function, you usually don't need to write the template arguments. The compiler deduces the type of T  from the function arguments. For example

````C++
max_of_three(1,2,3);


int max_of_three(const int a, const int b, const int c){
    return std::max({a,b,c})
}
````

The previous code produces an `int` specialization while the next code would generate a `double` specialization

````C++
max_of_three(1.0,2.3,3.5);

double max_of_thee(const double a, const double b, const double c){
    return std::max({a,b,c});
}
````

Be careful when you have mixed data types on the arguments, because the compiler can't pick one type, for example, in this case you have to specify the desired data type.

````C++
max_of_three(1, 2.0, 3);


max_of_thee<double>(1, 2.0, 3);
````


Because the code is generated during compilation, each specialization is as fast as hand-written code, and can be inlined and optimized. The trade-off is that compilation might take a bit longer. During compilation, the compiler determines the concrete template arguments for each use of a template. It infers them from how you call the template. Or if you've written the types explicitly, it uses those types. Once the types are determined, the compiler generates the specific functions or classes before the program runs. Template arguments affect the type system and overload resolution by determining which version of the code is generated. Function parameters are different. They are values that arrive when the program runs.

<img src="./IMG/Generic_Programming_01.png" width="400">

The code is already generated. Parameters simply provide the data that the code processes. The `auto` keyword lets the compiler write the exact type of a variable for you, which a relief when templates generate long iterator or Lambda types.  This allows template-heavy code to stay readable without sacrificing type safety. When the compilre generates code for specific types, it will generate it as if you has provided the final type yourself.

````C++
std::vector<int> nums = {1,2,3,4}
for(auto&n : nums){
    std::cout << n << " "; 
}
````

The `auto` keyword may also be used outside of templates as well, and it can save typing when the type is obvious or unimportant, such as for iterators, Lambdas, or long template types. It's still a good habit to write the type out in full when it helpss to communicate the intent of the code. Using the actual type when it's short and easy to read will also help with code readability and maintenance. Templates can be used to generate entire classes too. For example, the code in the next snippel generates a brand new class each time you use a new type. 

````C++
template <typename T>
class Box{
    T value_;
public:
    static inline int count = 0; /// one per specialization
    explicit Box(T v) : value_(v){ ++count;}
    const T& get() const {return value_;}
};
````
A Box int, stores an int, a Box std::string, sotres a std::string, and so on. Each specialization also gets its own static members. In this example, count is one static variable per specialization, so box and count is independent for Box std::string count, and the constructor increments the appropriate one. 


Because the generated classes have ordinary names at the call site, they work with function templates, with `auto`,a nd with the rest of the standard library. The copiler will almost always inline the tiny get accesor, so the abstaction costs essentially nothing at runtime beyond the code you actually use. 

````C++
// box.hpp
#pragma once
template<typename T>
    class Box{
        T value_; 
    public:
        explicit Box(T v) : value_(v){}
        const T& get() const{ return value_;}
};
````
The compiler must see the templates full body every time it instantiates it. Think of a template as a recipe. When you ask for Box int, the compile bakes a concrete Box of int by substituting int into that recipe. It can only do that if the full recipe, which is the template definition is visible to the compiler at that spot. 

If a cpp file only sees a forward declaration while the real template body is in a different cpp file, the compiler cannot generate the code it needs. In practice this often shows up as an unresolved external error from the linker. That's why we usually put both the template declaration and the definition ina header of file. then every cpp that includes the header can instantiate the template it uses. The trade-off is that the same definitions now appear in many translation units or each cpp file that includes the header. That's okay as long as the definitions are in line. Member functions defined inside the class, like the get function in this example, are implicitly inline for free functions or non member operators defined in headers at the inline keyword. This tells the compiler and linker that identical definitions coming from different files are one and the same, so they should be merged instead of duplicated. Starting in C++ 20, modules tive you a cleaner option. you may export the interface and place any template definitions that importers must use in the module interface or into an exported partition.


You can still hide non-exported helpers and other implementation details in the module's implementation unit. The compiler gets the full definitions through the modules metadata, so it can instantiate templates without textural has include duplication

Templates are powerful, but unconstrained templates accept any type, and compile time or run-time errors can often generate screens of diagnostic noise. 

Concepts add a named compile time predicates to a template's parameter list. The compiler will check the requirement before instantiation, and error messages will point directly to the call site. this is accomplised with no added runtime cost. 

````C++
// Constraint T to integral types only
template<std::integral T> T gcd(T a, Tb){
    while (b!=0){
        T tmp = b; 
        b = a % b; 
        a = tmp; 
    }
    return a; 
}

int main(){
    std::cout << gcd(42, 30) <<"\n"; // OK ints are integral 
    //std::cout << gcd(3.2,1.1); //error double are not integral
````

Here we'll see how **templates** enable zero-overhead, reusable code in C++:
- **Templates as blueprints:** Write one generic definitio; the compiler generates **specializations** for concrete types at **compile time**( no runtime cost, fully optimized).
- **Type deduction & guidance:** The compiler usually deduces template arguments from call sites; whe types mix or deduction fails, **specify the template argument** or **convert inputs**.
- **Functions vs parameters:** Template arguments are fixed at **compiled time; function parameters** are values supplied at **runtime**.
- **auto for readability:** Lets the compiler spell long templates types (iterators, lambdas) while keeping **type safety**.
- **Class templates:** One definition yields many types (e.g Box, Box std::string). **Static members are per-specialization**. Tiny accessors are typically **inlined**.
- **Where definitions live:** The compiler must see the ** full template definition** at instantiation - usually by **putting declarations and definition headers** (mark free functions inline).
- **Concepts:** Add compile-time **constraints** to template parameters for clearer errors and safer APIs - checked **before** instantiation, with **no runtime overhead**. 

## C++20 Modules - A Modern Atlernative to Headers

Before C++20, code reuse relied heavily on `#include` headers - which copy text into every translation unit that uses them. This causes: 
- **Slow compilation** Each `.cpp` repeatedly parses the same headers.
- **Fragile dependencies:** Preprocessor macros leak between files.
- **Template duplication:** Templates appear in every translation unit, increasing build times.

C++20 introduce **modules**, a system-level improvement to code organization and compilation: 
- A **module** is a compiled, importable unit that exports selected declarations.
- When a file *imports* a module, it reads the compiled module metadata - *not raw text.*
- Templates inside modules can be defined once and compiled into that module's metadata. This eliminates textual duplication while preserving type safety and visibility rules.
- You separate your code into:
  
   - **Interface units** (export module Math;) - what users import.
   - **Implementation units** (module Math;) - internal details that stay hidden.
   - **Partitions** - Optional modular subcomponents of larget systems.

C++20 Modules offer significant advantages over the traditional header-based inclusion model, making them a compelling choice for modern c++ development. The primary reason to use C++20 Modules include: 

- **Faster Compilation Times:** Modules are compiled once into a binary interface (BMI) and then reused, eliminating the need for redundant parsing of header files in every translation unit that includes them. This drastically redues overall build times, especially in large projects.

- **Improved Encapsulation and Dependency Management:** Modules provide explicit control over what symbols are exported, preventing unintended dependencies and reducing the risk of name collisions (e.g. ODR violations). This leads to cleaner, more maintainable code with better-defined interfaces.
- **Elimination of Macro Pollution and Include Order Issues:** Macros are not automatically exported from modules, preventing them from "leaking" into other parts of the codebase and causing unexpected behavior. Additionally, the order in which modules are imported does not affect their behavior, unlike with header files where include order can be critical.

- **Enhanced Tooling and IDE support:** The structured nature of modules provides better information for compilers and IDEs, leading to more accurate error reporting, improved code completion, and more efficient analysis.

- **Alignment with Modern C++ Practices:** Modules represent a fundamental shift in how C++ code is organized and compiled, moving towards a more robust and scalable model. Adopting modules positions projects for future advancements in the language and ecosystem. 